# Interactive (live): watch a gravel river adjust in real time

Unlike the [equilibrium demo](interactive_single_segment.ipynb), this one **runs continuously**. While it's playing, drag the sliders and watch the long profile respond *transiently*: turn **sediment** up and it aggrades and steepens; drop **base level** and an incision wave climbs upstream. GRLP steps forward in your browser (Pyodide), frame by frame.

Run all cells, then use the sliders and the **Running / Reset** buttons.

In [ ]:
%pip install -q grlp networkx ipywidgets ipympl

In [ ]:
%matplotlib widget
import asyncio

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

import grlp

YEAR = 31556926.   # seconds per year
DT = 10 * YEAR     # 10 years advanced per animation frame


def make_equilibrium(Qw, Qs, zbl):
    """A single segment started at steady state for the given inputs."""
    lp = grlp.LongProfile()
    lp.basic_constants()
    lp.bedload_lumped_constants()
    lp.set_hydrologic_constants()
    lp.set_x(dx=1000., nx=60, x0=1000.)
    lp.set_z(S0=-1e-2, z1=zbl)
    lp.set_Q(Qw)
    lp.set_B(100.)
    lp.set_niter(3)
    lp.set_uplift_rate(0.)
    lp.set_z_bl(zbl)
    lp.set_Qs_input_upstream(Qs)
    lp.evolve_threshold_width_river(nt=10, dt=1e13)
    return lp


Q0, QS0, ZBL0 = 100., 0.02, 0.
lp = make_equilibrium(Q0, QS0, ZBL0)
sim_t = 0.

plt.ioff()                                  # embed the canvas ourselves, below
fig, ax = plt.subplots(figsize=(8, 4))
fig.canvas.header_visible = False
(line,) = ax.plot(lp.x / 1000., lp.z, lw=3)
(bl_line,) = ax.plot([lp.x.min() / 1000., lp.x_ghost_downstream / 1000.],
                     [ZBL0, ZBL0], color='0.6', ls='--', lw=1)
ax.set_xlim(lp.x.min() / 1000., lp.x.max() / 1000.)
ax.set_ylim(-120, 1300)
ax.set_xlabel('Downstream distance [km]')
ax.set_ylabel('Elevation [m]')

In [ ]:
Qw  = widgets.FloatSlider(value=Q0,   min=20, max=600, step=20,
                          description='Water discharge $Q$ [m³/s]',
                          style={'description_width': '340px'},
                          layout=widgets.Layout(width='760px'))
Qs  = widgets.FloatSlider(value=QS0,  min=0.005, max=0.06, step=0.005,
                          description='Bed-load sediment input $Q_s$ [m³/s]', readout_format='.3f',
                          style={'description_width': '340px'},
                          layout=widgets.Layout(width='760px'))
zbl = widgets.FloatSlider(value=ZBL0, min=-100, max=100, step=5,
                          description='Base level [m]',
                          style={'description_width': '340px'},
                          layout=widgets.Layout(width='760px'))
play  = widgets.ToggleButton(value=True, description='Running', icon='play')
reset = widgets.Button(description='Reset', icon='refresh')


def on_play(change):
    play.description = 'Running' if change['new'] else 'Paused'
    play.icon = 'play' if change['new'] else 'pause'
play.observe(on_play, 'value')


def do_reset(_):
    global lp, sim_t
    lp = make_equilibrium(Qw.value, Qs.value, zbl.value)
    sim_t = 0.
    line.set_ydata(lp.z)
    fig.canvas.draw_idle()
reset.on_click(do_reset)


async def run():
    global sim_t
    while True:
        if play.value:
            # the sliders ARE the current boundary conditions, read live
            lp.set_Q(Qw.value)
            lp.set_Qs_input_upstream(Qs.value)
            lp.set_z_bl(zbl.value)
            lp.evolve_threshold_width_river(nt=1, dt=DT)
            sim_t += DT
            line.set_ydata(lp.z)
            bl_line.set_ydata([zbl.value, zbl.value])
            ax.set_title('t = %.1f kyr' % (sim_t / (1000. * YEAR)))
            fig.canvas.draw_idle()
        await asyncio.sleep(0.05)


asyncio.ensure_future(run())
widgets.VBox([fig.canvas, widgets.HBox([play, reset]), Qw, Qs, zbl])